In [41]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

In [42]:
caminho_pdf = ("documents/inteligencia_artificial.pdf")

loader = PyPDFLoader(str(caminho_pdf))

documentos = loader.load()

print(f"Total de páginas carregadas: {len(documentos)}")
print(documentos[0].page_content[:500])

Total de páginas carregadas: 119
1 
 
 
 
 
 
 
 
 
 
 
INTELIGÊNCIA ARTIFICIAL 
Vinicius Ponte Machado


In [35]:
divisor = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

trechos = divisor.split_documents(documentos)

print(f"Quantidade de trechos: {len(trechos)}")
print(trechos[0].page_content)

Quantidade de trechos: 188
1 
 
 
 
 
 
 
 
 
 
 
INTELIGÊNCIA ARTIFICIAL 
Vinicius Ponte Machado


In [36]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vetor_teste = embeddings.embed_query(
    "O que é inteligência artificial?"
)

print(f"Dimensões do vetor: {len(vetor_teste)}")
print(vetor_teste[:10])

Dimensões do vetor: 768
[0.021165347, 0.08557707, -0.12836279, -0.023675874, 0.0503834, 0.0426596, -0.007680998, 0.012495923, -0.046243016, 0.03442699]


In [37]:
banco_vetorial = FAISS.from_documents(
    documents=trechos,
    embedding=embeddings
)

print("Banco vetorial criado com sucesso!")

Banco vetorial criado com sucesso!


In [38]:
pergunta = "Do que se trata o documento?"

resultados = banco_vetorial.similarity_search(
    pergunta,
    k=3
)

for i, documento in enumerate(resultados, start=1):
    print(f"\n--- Resultado {i} ---")
    print(documento.page_content[:500])


--- Resultado 1 ---
• Completeza: sempre encontra uma solução se esta ex istir; 
• Otimização: encontra a solução ótima 
• Complexidade de tempo: tempo gasto para encontrar uma solução; 
• Complexidade de espaço: memória necessária para en contrar uma solução. 
                                            
1 Árvore, no contexto da programação e ciência da co mputação, é uma estrutura de dados que herda 
as características das topologias em árvore. Concei tualmente diferente das listas encadeadas, em 
que os dados se

--- Resultado 2 ---
92 
• As ações Esquerda e Direita movem o agente nesta d ireções; exceto quando isto 
leva o agente para fora do ambiente (fica parado) 
• Ações disponíveis: Esquerda, Direita, Aspirar, NoO P (não faz nada)  
• O agente percebe: a sua posição e se nessa posição  existe sujeira 
 
Outras questões também deve ser levadas em consider ação quando se fala 
no conceito de agentes como por exemplo a racionali dade e autonomia. Um agente 
racional, na visão da 

In [39]:
modelo = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

recuperador = banco_vetorial.as_retriever(
    search_kwargs={"k": 3}
)

qa = RetrievalQA.from_chain_type(
    llm=modelo,
    chain_type="stuff",
    retriever=recuperador,
)

print("RetrievalQA configurado!")

RetrievalQA configurado!


In [40]:
pergunta = input("Digite sua pergunta sobre o documento: ")

resultado = qa.invoke({
    "query": pergunta
})

print("\nRESPOSTA:")
print(resultado["result"])


RESPOSTA:
Não sei como construir um jogador para o jogo da Velha.
